# Bài 6 — Đếm xe qua vạch: LineZone

**Mục tiêu:** Đếm xe theo 2 chiều Vào/Ra — **nhìn số đếm nhảy realtime ngay trên cửa sổ**.

## 0. Chuẩn bị (asset video + display.py dùng chung)

In [10]:
!pip install -q supervision ultralytics "supervision[assets]"

In [11]:
from supervision.assets import download_assets, VideoAssets

download_assets(VideoAssets.VEHICLES)
print(VideoAssets.VEHICLES.value)  # "vehicles.mp4"

[2026-08-18 16:22:03] [INFO] supervision.assets.downloader - Downloading vehicles.mp4 assets


  0%|          | 0/35345757 [00:00<?, ?it/s]

vehicles.mp4


In [12]:
%%writefile display.py
# display.py — hàm hiển thị dùng chung cho toàn giáo trình
import cv2

WINDOW_NAME = "Supervision - Live"
MAX_DISPLAY_WIDTH = 1280   # thu nhỏ frame cho vừa màn hình (chỉ để XEM, không ảnh hưởng xử lý)


def show_frame(frame, window_name: str = WINDOW_NAME, wait: int = 1) -> bool:
    """Hiện frame lên cửa sổ. Trả về False nếu người dùng bấm Q/ESC (muốn thoát).

    wait=1  -> dùng cho video (hiện liên tục, không chặn)
    wait=0  -> dùng cho ảnh tĩnh (dừng lại chờ bấm phím bất kỳ)
    """
    h, w = frame.shape[:2]
    if w > MAX_DISPLAY_WIDTH:                      # thu nhỏ để vừa màn hình
        scale = MAX_DISPLAY_WIDTH / w
        frame = cv2.resize(frame, (int(w * scale), int(h * scale)))

    cv2.imshow(window_name, frame)
    key = cv2.waitKey(wait) & 0xFF
    if key in (ord("q"), ord("Q"), 27):            # Q hoặc ESC -> thoát
        return False
    return True


def close_windows():
    cv2.destroyAllWindows()

Writing display.py


## 6.1. Nguyên lý

`LineZone` là 1 đoạn thẳng ảo. Khi anchor point của 1 track đi từ bên này sang bên kia vạch → tăng `in_count`/`out_count`. **Bắt buộc phải có `tracker_id`** (Bài 5).

## 6.2. Công cụ tìm tọa độ vạch — click chuột đọc tọa độ ngay trên cửa sổ

In [13]:
import cv2
import supervision as sv

frame = next(sv.get_video_frames_generator("vehicles.mp4"))
small = cv2.resize(frame, None, fx=0.5, fy=0.5)   # thu nhỏ; nhớ nhân lại 2 khi dùng

def on_mouse(event, x, y, flags, param):
    if event == cv2.EVENT_LBUTTONDOWN:
        print(f"Toa do (da thu nho): ({x}, {y})  ->  goc: ({x*2}, {y*2})")

cv2.namedWindow("Click de lay toa do - Q de thoat")
cv2.setMouseCallback("Click de lay toa do - Q de thoat", on_mouse)
while True:
    cv2.imshow("Click de lay toa do - Q de thoat", small)
    if cv2.waitKey(20) & 0xFF in (ord("q"), 27):
        break
cv2.destroyAllWindows()

## 6.3. Pipeline đếm xe qua vạch — giữ khung Bài 4/5

In [14]:
import numpy as np
from collections import defaultdict
from ultralytics import YOLO
from display import show_frame, close_windows

SOURCE_VIDEO = "vehicles.mp4"
TARGET_VIDEO = "bai6_output.mp4"
VEHICLE_CLASSES = [2, 3, 5, 7]
MAX_FRAMES = 300   # test nhanh trước; đặt None để chạy hết video

model = YOLO("yolov8n.pt")
CLASS_NAMES = model.names   # dict {class_id: ten_class} — dùng thay vì detections.data["class_name"]
# (data["class_name"] bị ByteTrack loại bỏ sau khi update_with_detections, nên không dùng được nữa)
video_info = sv.VideoInfo.from_video_path(SOURCE_VIDEO)

tracker = sv.ByteTrack(frame_rate=video_info.fps)

# Vạch ngang giữa khung hình — chỉnh lại theo tọa độ đọc được ở 6.2
START = sv.Point(0, video_info.height // 2)
END = sv.Point(video_info.width, video_info.height // 2)

line_zone = sv.LineZone(
    start=START, end=END,
    triggering_anchors=[sv.Position.BOTTOM_CENTER],
    minimum_crossing_threshold=2,   # cần 2 frame xác nhận, chống đếm trùng do jitter
)
line_annotator = sv.LineZoneAnnotator(
    thickness=2, text_scale=0.8,
    custom_in_text="Vao", custom_out_text="Ra",
)

box_annotator = sv.BoxAnnotator(thickness=2, color_lookup=sv.ColorLookup.TRACK)
label_annotator = sv.LabelAnnotator(text_scale=0.5, color_lookup=sv.ColorLookup.TRACK)

counts = defaultdict(int)   # thống kê chi tiết theo loại xe

## 6.4. Bảng thống kê realtime + hàm xử lý từng frame

In [15]:
def draw_stats(frame, stats: dict, origin=(20, 50)):
    """Vẽ bảng thống kê nền đen mờ ở góc trên trái."""
    if not stats:
        return frame
    x, y = origin
    line_h = 40
    h = line_h * (len(stats) + 1)
    overlay = frame.copy()
    cv2.rectangle(overlay, (x - 10, y - 35), (x + 380, y - 35 + h), (0, 0, 0), -1)
    frame[:] = cv2.addWeighted(overlay, 0.6, frame, 0.4, 0)
    for i, (k, v) in enumerate(stats.items()):
        cv2.putText(frame, f"{k}: {v}", (x, y + i * line_h),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255, 255, 255), 2)
    return frame


def process_frame(frame: np.ndarray) -> np.ndarray:
    results = model(frame, imgsz=640, verbose=False)[0]
    detections = sv.Detections.from_ultralytics(results)
    detections = detections[np.isin(detections.class_id, VEHICLE_CLASSES)]
    detections = tracker.update_with_detections(detections)

    #  Cập nhật bộ đếm (gọi trigger 1 lần duy nhất)
    crossed_in, crossed_out = line_zone.trigger(detections)
    for cid in detections.class_id[crossed_in]:
        counts[f"{CLASS_NAMES[cid]}_in"] += 1
    for cid in detections.class_id[crossed_out]:
        counts[f"{CLASS_NAMES[cid]}_out"] += 1

    labels = [f"#{tid} {CLASS_NAMES[cid]}" for tid, cid
              in zip(detections.tracker_id, detections.class_id)]

    annotated = frame.copy()
    annotated = box_annotator.annotate(annotated, detections)
    annotated = label_annotator.annotate(annotated, detections, labels=labels)
    annotated = line_annotator.annotate(annotated, line_counter=line_zone)
    annotated = draw_stats(annotated, counts)
    return annotated

In [16]:
tracker.reset()

sink = sv.VideoSink(target_path=TARGET_VIDEO, video_info=video_info)
sink.__enter__()
try:
    for i, frame in enumerate(sv.get_video_frames_generator(SOURCE_VIDEO)):
        if MAX_FRAMES is not None and i >= MAX_FRAMES:
            break
        annotated = process_frame(frame)
        sink.write_frame(annotated)
        if not show_frame(annotated):
            print("Nguoi dung bam Q - dung som.")
            break
finally:
    sink.__exit__(None, None, None)
    close_windows()

print(f"Tong vao: {line_zone.in_count} | Tong ra: {line_zone.out_count}")
print("Chi tiet:", dict(counts))

Tong vao: 2 | Tong ra: 1
Chi tiet: {'car_in': 2, 'car_out': 1}


## Checkpoint Bài 6

Trên cửa sổ live: vạch đếm hiện số Vào/Ra nhảy realtime khi xe cắt vạch, góc trái có bảng thống kê `car_in / car_out / truck_in / ...` cập nhật liên tục.